# 10. 대화형 거래 조사 Agent

첫 설정 셀을 실행한 뒤 마지막 대화 입력 셀에서 문장을 입력한다. `종료`로 입력을 마친다. 대화와 최근 12개 메시지를 Groq에 전달하며, DB의 계좌 식별자와 원시 이력은 도구 결과에서 제외한다. 조회 요청은 조회만 수행하고 조사 요청은 새 보고서를 저장한다. API는 로컬 시연용으로 사용한다. 커널을 재시작하면 대화 이력을 초기화한다.

중간 자동 시연 셀은 실제 조회·조사·보고서 재조회를 수행하며 조사 보고서 한 건을 저장한다. 전체 실행 시 마지막 셀에서 사용자 입력을 기다린다.

In [1]:
import sys
from pathlib import Path
from fastapi.testclient import TestClient

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fds_agent.api.main import app

client = TestClient(app)
history = []

def ask(message):
    """최근 대화를 전달하고 실제 사용한 도구와 답변을 표시한다."""
    response = client.post("/v1/chat", json={"message": message, "history": history[-12:]})
    response.raise_for_status()
    result = response.json()
    print("사용자:", message)
    print("Agent:", result["answer"])
    print("처리 상태:", result["status"])
    print("사용 도구:", [action["tool"] for action in result["actions"]])
    if result["status"] == "completed":
        history.extend([{"role": "user", "content": message},
                        {"role": "assistant", "content": result["answer"]}])
    return result


## 실제 연속 대화 시연

검색 결과를 대화 맥락으로 전달하여 거래 조사와 저장 보고서 재조회를 검증한다.

In [2]:
first = ask("저장된 의심 거래를 최신 순으로 1건 찾아줘. 거래 ID도 알려줘.")
assert first["status"] == "completed"
assert any(a["tool"] == "search_transactions" for a in first["actions"])
second = ask("방금 찾은 그 거래를 조사하고 보고서를 저장해줘. 보고서 ID도 알려줘.")
assert second["status"] == "completed"
investigations = [a["result"] for a in second["actions"] if a["tool"] == "investigate_transaction"]
assert investigations and "report_id" in investigations[-1]
report_id = investigations[-1]["report_id"]
third = ask(f"보고서 {report_id}의 결과를 다시 조회해줘. 새 조사는 하지 마.")
assert third["status"] == "completed"
assert any(a["tool"] == "get_investigation" for a in third["actions"])
assert not any(a["tool"] == "investigate_transaction" for a in third["actions"])
print("검색 → 후속 조사 → 저장 보고서 재조회 흐름을 확인한다.")


사용자: 저장된 의심 거래를 최신 순으로 1건 찾아줘. 거래 ID도 알려줘.
Agent: 최신 의심 거래의 ID는 684af5fb-fb45-5001-af51-1ee004446edf이며, 이를 확인한다.
처리 상태: completed
사용 도구: ['search_transactions']


사용자: 방금 찾은 그 거래를 조사하고 보고서를 저장해줘. 보고서 ID도 알려줘.
Agent: 조사 결과, 보고서 ID는 1802b6dd-9f0a-4f16-8e57-8be24e4724ac이며, 거래 ID는 684af5fb-fb45-5001-af51-1ee004446edf이다. 점수와 판단은 도구 결과에 따르며, 추가적인 조치는 현재 제공되지 않는다.
처리 상태: completed
사용 도구: ['investigate_transaction']


사용자: 보고서 1802b6dd-9f0a-4f16-8e57-8be24e4724ac의 결과를 다시 조회해줘. 새 조사는 하지 마.
Agent: 조회 결과, 보고서 ID는 1802b6dd-9f0a-4f16-8e57-8be24e4724ac이며, 해당 거래 ID는 684af5fb-fb45-5001-af51-1ee004446edf이다. 보고서 상태는 review_required이며, 위험 점수는 0.999999999999995, 임계값은 0.00635036124517738이다. 주요 발견 사항은 model_alert, full_balance_transfer, prior_incoming이며, 각각의 증거 ID가 포함되어 있다. 설명은 LLM 보조 방식으로 생성되었다. 이를 확인한다.
처리 상태: completed
사용 도구: ['get_investigation']
검색 → 후속 조사 → 저장 보고서 재조회 흐름을 확인한다.


## 직접 대화

아래 셀을 실행하고 입력창에 `의심 거래 3건 찾아줘` 등을 입력한다. 대화 기록을 초기화하려면 `history.clear()`를 실행한다. 입력 내용도 Groq로 전달하므로 비밀번호와 API 키는 입력하지 않는다.

In [ ]:
while True:
    message = input("요청을 입력한다. 종료하려면 '종료'를 입력한다: ").strip()
    if message in {"종료", "exit", "quit"}:
        break
    if message:
        result = ask(message)
